# 03 · Hill climbing

Greedy never goes back on a decision. **Hill climbing** does: it takes a finished schedule and keeps making small changes, called **moves**.

1. Start from the greedy schedule.
2. Propose one random move (shift a task by a day, swap a technician, ...).
3. If the score goes **up**, keep it. If not, undo it.
4. Repeat until many tries in a row bring no improvement.

Think of climbing a hill in thick fog: you can only feel the ground around your feet, so you always step uphill, and you stop when every step around you goes down. The catch: that might be a small hill, not the highest one. That's called a **local optimum**, and it's what simulated annealing in notebook 04 is built to escape.

You write the **shift move** and the **accept rule** (**YOUR TURN** cells).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research').is_dir())
sys.path.insert(0, str(ROOT))

import random
import time
from collections import Counter
from datetime import timedelta

import matplotlib.pyplot as plt
import pandas as pd

from research.evaluate import score, summary, validate
from research.local_search import Change, Schedule, insert_move, random_placed_task, replace_move
from research.reference import hill_climbing as reference
from research.reference.greedy import run_greedy
from research.synthetic import Scenario, generate

data = generate(seed=1)
greedy_result = run_greedy(**data)
print('Greedy score:', score(greedy_result, data).total)

## Your toolbox

- `Schedule(data, result)` holds one complete schedule you can change. `schedule.placements[task_id]` is that task's `(team, start)`, and `schedule.total()` is its score.
- A move **proposes** a change as `Change(task_id, before, after)`, where `before` and `after` are each `(team, start)`.
- `schedule.apply(change)` makes the change if it breaks no hard rule, and returns `False` (changing nothing) if it would. `schedule.undo(change)` reverses it.
- `random_placed_task(schedule, rng)` picks a random scheduled task id.

Here's a ready-made move, `replace_move` (swap one technician), being tried and undone:

In [ ]:
schedule = Schedule(data, greedy_result)
rng = random.Random(0)
print('Score now:', schedule.total())

change = replace_move(schedule, rng)
task = schedule.tasks[change.task_id]
print(task['doc_no'], ':', [t['name'] for t in change.before[0]], '->', [t['name'] for t in change.after[0]])

if schedule.apply(change):
    print('Applied. Score now:', schedule.total())
    schedule.undo(change)
    print('Undone.  Score now:', schedule.total())
else:
    print('Refused: the new technician is busy on one of those days.')

## Move 1: shift a task

Pick a random scheduled task and propose the **same team** starting 1 to `max_shift` days earlier or later. Don't worry about whether the new days are free or too early: `schedule.apply()` checks that and refuses bad changes.

In [ ]:
MAX_SHIFT_DAYS = 3


# YOUR TURN
def shift_move(schedule, rng, max_shift=MAX_SHIFT_DAYS):
    """Propose moving a random task 1 to max_shift days earlier or later, with the same team.

    Steps:
    1. task_id = random_placed_task(schedule, rng); return None if it is None.
    2. team, start = schedule.placements[task_id]
    3. Pick a shift that isn't 0: rng.choice([n for n in range(-max_shift, max_shift + 1) if n != 0])
    4. return Change(task_id, (team, start), (team, start + timedelta(days=shift)))
    """
    raise NotImplementedError('Write shift_move')

In [ ]:
# Check: 200 proposals must keep the team, move the start by 1..MAX_SHIFT_DAYS days, and never by 0.
check = Schedule(data, greedy_result)
rng = random.Random(1)
shifts = Counter()
for _ in range(200):
    change = shift_move(check, rng)
    assert change.before[0] == change.after[0], 'the team must stay the same'
    days = (change.after[1] - change.before[1]).days
    assert days != 0 and abs(days) <= MAX_SHIFT_DAYS, f'shift of {days} days is out of range'
    shifts[days] += 1
print('shift_move looks right. Shifts proposed:', dict(sorted(shifts.items())))

## The accept rule

After a move, the loop asks: keep it or undo it? Hill climbing keeps a move only when the new score is **higher** than the current one.

This one line is the whole difference between hill climbing and simulated annealing: in notebook 04 you'll change it to *sometimes* accept a worse score.

In [ ]:
# YOUR TURN
def accept(current, candidate):
    """True to keep the move. current = score before the move, candidate = score after it."""
    raise NotImplementedError('Write accept')

In [ ]:
# Check
assert accept(100, 120) is True, 'a better score must be kept'
assert accept(100, 80) is False, 'a worse score must be undone'
assert accept(100, 100) is False, 'an equal score is not an improvement'
print('accept looks right.')

## The hill climbing loop

Each try picks a move at random (shift and replace often, insert now and then), applies it, and asks your `accept` whether to keep it. It stops after `max_stall` tries in a row without an improvement.

`insert_move` tries to fit in a task the greedy left unscheduled.

In [ ]:
MOVES = {'shift': shift_move, 'replace': replace_move, 'insert': insert_move}
MOVE_WEIGHTS = {'shift': 4, 'replace': 4, 'insert': 1}
MAX_STALL = 2000


def my_hill_climb(data, start_result, seed=0, max_stall=MAX_STALL, max_iterations=50_000):
    rng = random.Random(seed)
    names = list(MOVES)
    weights = [MOVE_WEIGHTS[name] for name in names]

    schedule = Schedule(data, start_result)
    current = schedule.total()
    history, accepted, stall = [current], Counter(), 0

    for _ in range(max_iterations):
        name = rng.choices(names, weights=weights)[0]
        change = MOVES[name](schedule, rng)
        if change is not None and schedule.apply(change):
            candidate = schedule.total()
            if accept(current, candidate):
                stall = 0 if candidate > current else stall + 1
                current = candidate
                accepted[name] += 1
            else:
                schedule.undo(change)
                stall += 1
        else:
            stall += 1          # the move broke a hard rule, or there was nothing to move
        history.append(current)
        if stall >= max_stall:
            break
    return schedule, history, accepted


started = time.perf_counter()
schedule, history, accepted = my_hill_climb(data, greedy_result)
result = schedule.to_result('my_hill_climbing', started)

print('Problems:', validate(result, data) or 'none')
print(f"Greedy {score(greedy_result, data).total}  ->  hill climbing {score(result, data).total}")
print(f'{len(history) - 1} tries in {result["runtime_ms"]:.0f} ms; improvements by move: {dict(accepted)}')

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(history)
plt.xlabel('try')
plt.ylabel('score')
plt.title('Hill climbing: the score only ever goes up, then flattens')
plt.show()

In [ ]:
pd.DataFrame([summary(greedy_result, data), summary(result, data)], index=['greedy', 'hill climbing'])

## Stuck on a local optimum

Run the same problem with five different random seeds. If hill climbing always found the best schedule, all five would end on the same score. They usually don't: each run climbs a different hill and stops at its top.

In [ ]:
plt.figure(figsize=(9, 3.5))
finals = []
for run_seed in range(5):
    _schedule, run_history, _accepted = my_hill_climb(data, greedy_result, seed=run_seed)
    finals.append(run_history[-1])
    plt.plot(run_history, label=f'seed {run_seed}: {run_history[-1]:.0f}')
plt.xlabel('try')
plt.ylabel('score')
plt.legend()
plt.show()
print('Final scores:', finals)

## Twenty problems, side by side

In [ ]:
rows = []
for seed in range(20):
    problem = generate(seed=seed)
    start = run_greedy(**problem)
    started = time.perf_counter()
    mine = my_hill_climb(problem, start)[0].to_result('my_hill_climbing', started)
    ref = reference.run_hill_climbing(**problem)
    greedy_score = score(start, problem).total
    rows.append({
        'seed': seed,
        'valid': not validate(mine, problem),
        'greedy': greedy_score,
        'your hill climbing': score(mine, problem).total,
        'reference': score(ref, problem).total,
        'gain': round(score(mine, problem).total - greedy_score, 1),
        'unscheduled': f"{len(start['unscheduled'])} -> {len(mine['unscheduled'])}",
        'ms': mine['runtime_ms'],
    })
comparison = pd.DataFrame(rows)
comparison

In [ ]:
comparison[['greedy', 'your hill climbing', 'gain', 'ms']].describe().round(1)

## Experiments

Change something, re-run, and write what you notice in a markdown cell or in `notes/personal_notes.md`.

1. **Shift size.** Try `MAX_SHIFT_DAYS` = 1 and 7. Bigger jumps reach further but are refused more often. Which wins?
2. **Which moves matter?** Remove `'replace'` from `MOVES` (and from `MOVE_WEIGHTS`), then remove `'shift'` instead. Which move does most of the work?
3. **Patience.** Try `MAX_STALL` = 200 and 10000. How much score do you gain for how much extra time?
4. **A new move (challenge).** Write `swap_teams_move`: pick two scheduled tasks and swap their teams. Hint: a swap changes two tasks, but a `Change` covers one, so apply two `Change`s and undo both if either is refused.
5. **Equal scores.** Change `accept` to `candidate >= current`. The search can now drift sideways across flat ground. Does it find better schedules?